# ПР-13. Многослойный персептрон на табличных данных

Обучить MLP-классификатор, изучить влияние архитектуры и регуляризации и сравнить его с градиентным бустингом.

**Данные:** см. README проекта. Выполняйте ячейки сверху вниз; фиксируйте результаты в отчёте.

## 1. Подготовка данных

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
data = pd.read_csv(ROOT / 'data' / 'classification_breast_cancer.csv')
X = data.drop(columns=['target', 'target_name'])
y = data['target'].astype(int)  # 0 = malignant, 1 = benign
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print('train/test:', X_train.shape, X_test.shape)


## 1. Подбор MLP

Нейросеть требует масштабирования. Pipeline выполняет его внутри каждого CV-фолда. Для сходимости задан увеличенный лимит итераций и ранняя остановка.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold,GridSearchCV
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
pipe=Pipeline([('scale',StandardScaler()),('mlp',MLPClassifier(max_iter=1000,early_stopping=True,
                                                   n_iter_no_change=20,random_state=42))])
search=GridSearchCV(pipe,{'mlp__hidden_layer_sizes':[(32,),(64,),(64,32)],
                          'mlp__alpha':[0.0001,0.01,0.1]},
                    scoring='f1_macro',cv=cv,n_jobs=-1)
search.fit(X_train,y_train)
pred=search.predict(X_test)
print(search.best_params_,'CV F1 macro=',round(search.best_score_,3))
print(classification_report(y_test,pred,target_names=['malignant','benign']))

## 2. Сравнение с бустингом

Сравните не только итоговый балл, но также затраты на настройку и устойчивость CV. Используйте одинаковые разбиения.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score
boost=HistGradientBoostingClassifier(max_iter=100,random_state=42)
boost_cv=cross_val_score(boost,X_train,y_train,cv=cv,scoring='f1_macro').mean()
boost.fit(X_train,y_train); boost_test=f1_score(y_test,boost.predict(X_test),average='macro')
print('MLP CV/test:',round(search.best_score_,3),round(f1_score(y_test,pred,average='macro'),3))
print('Boosting CV/test:',round(boost_cv,3),round(boost_test,3))

## Контрольные вопросы и вывод

1. Почему масштабирование особенно важно для MLP?
2. Что означают batch и epoch? Как соотносится max_iter с эпохами обучения?
3. Почему нейронная сеть не обязана превосходить бустинг на табличных данных?
4. Как alpha и early stopping ограничивают переобучение?

**В отчёте:** выбранная архитектура, параметры, CV/test метрики, сравнение с бустингом и краткий вывод о целесообразности MLP.